# 📈 Clenow Momentum Strategy — Analysis Notebook

This notebook walks through our enhanced Clenow Trend strategy step by step:

1. **Regime Filter** — Are broad markets healthy? (SPY / IJH / IJR breadth)
2. **Entry Filter & Ranking** — Score the universe, only keep stocks above their 200-SMA
3. **Inverse-Vol Sizing** — Allocate more to lower-vol names (2–10% per position)
4. **Exit Logic** — When do we sell? (rank drops or price breaks below 100-SMA)
5. **Deep Dive** — Regression fits, score distributions, relative strength

In [ ]:
# ── Imports & Setup (run this cell first) ──

import sys, os, importlib
from dataclasses import dataclass
from typing import Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import yfinance as yf
from scipy.stats import linregress

# Point Python at our src/ folder so we can import the strategy module
SRC_PATH = os.path.abspath("../../../src")
if SRC_PATH not in sys.path:
    sys.path.insert(0, SRC_PATH)

import strategies.momentum.clenow_trend as ct
importlib.reload(ct)  # always pick up latest edits without restarting kernel

# ── Config ──
TOP_N = ct.DEFAULT_TOP_N          # 20
UNIVERSE   = "../../../universe.csv"
BUDGET     = 10_000               # example budget for sizing demo

print("✅ Setup complete")

## Step 1 — Regime Filter

We check if the broad market is healthy before buying anything new. We download SPY (large cap), IJH (mid cap), and IJR (small cap). If at least **2 out of 3** are trading above their 200-day SMA, we consider it "risk-on" and proceed. Otherwise, we skip new entries.

In [ ]:
# Run the regime check from our strategy module
regime_on = ct.check_regime()
print(f"Regime: {'✅ RISK-ON' if regime_on else '🛑 RISK-OFF (no new buys)'}\n")

# Plot each ETF vs its 200-day SMA so we can visually confirm
etf_data = yf.download(list(ct.REGIME_ETFS), period="1y", group_by="ticker", progress=False, threads=True)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, etf in zip(axes, ct.REGIME_ETFS):
    close = etf_data[etf]["Close"].dropna()
    sma = close.rolling(ct.REGIME_SMA_PERIOD).mean()
    above = float(close.iloc[-1]) > float(sma.dropna().iloc[-1]) if len(sma.dropna()) > 0 else False

    ax.plot(close.index, close, label="Close")
    ax.plot(sma.index, sma, "--", color="orange", label="200-SMA")
    ax.set_title(f"{etf}  {'✅' if above else '❌'}")
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

fig.suptitle("Regime Filter: ETFs vs 200-day SMA", fontweight="bold")
plt.tight_layout()
plt.show()

## Step 2 — Score Universe, Entry Filter & Ranking

We score every stock in our universe using the **Clenow Score** (annualized log-price slope × R²). But before scoring, we filter out any stock whose current price is **below its 200-day SMA** — no point buying something in a downtrend.

The top 20 by score become our buy candidates.

In [ ]:
# score_universe() handles the 200-SMA entry filter internally
scores_df = ct.score_universe(universe_file=UNIVERSE)
picks = scores_df.head(TOP_N)["Symbol"].tolist()

print(f"Scored {len(scores_df)} stocks that passed the 200-SMA entry filter")
print(f"\nOur top {TOP_N} picks:")
print(scores_df.head(TOP_N).to_string(index=False))

# Normalized price chart — rebase everything to 100 so we can compare % moves
if picks:
    price_data = yf.download(picks, period="6mo", progress=False)["Close"]
    normed = (price_data / price_data.iloc[0]) * 100

    plt.figure(figsize=(12, 6))
    for col in normed.columns:
        plt.plot(normed.index, normed[col], label=col)
    plt.title(f"Top {len(picks)} Momentum Stocks — 6mo Performance (rebased to 100)")
    plt.ylabel("Performance Index")
    plt.legend(ncol=4, fontsize=8)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

## Step 3 — Inverse-Volatility Sizing

Instead of giving equal $ to every stock, we give **more weight to lower-volatility** names. This means smoother, steadier trends get bigger positions — which fits the whole Clenow philosophy.

Weights are clamped between **2% and 10%** per position so no single stock dominates.

In [ ]:
# Compute inverse-vol weights for our top picks
top = scores_df.head(TOP_N).set_index("Symbol")
weights = ct._inverse_vol_weights(top["Vol20"])

sizing = top[["Score", "Close", "Vol20"]].copy()
sizing["Weight %"] = weights * 100
sizing["Notional ($)"] = weights * BUDGET  # how much $ each stock gets

print(f"Sizing for a ${BUDGET:,.0f} budget:\n")
print(sizing[["Score", "Vol20", "Weight %", "Notional ($)"]].to_string(float_format="%.2f"))
print(f"\nTotal weight: {weights.sum():.4f}  |  Total notional: ${(weights * BUDGET).sum():,.2f}")

# Side-by-side: weights vs volatility
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.bar(sizing.index, sizing["Weight %"], color="steelblue")
ax1.axhline(ct.MIN_WEIGHT * 100, color="red", ls="--", label=f"Min {ct.MIN_WEIGHT*100:.0f}%")
ax1.axhline(ct.MAX_WEIGHT * 100, color="red", ls="--", label=f"Max {ct.MAX_WEIGHT*100:.0f}%")
ax1.set_title("Weight per Stock")
ax1.set_ylabel("Weight (%)")
ax1.tick_params(axis="x", rotation=60)
ax1.legend()
ax1.grid(True, axis="y", alpha=0.3)

ax2.bar(sizing.index, sizing["Vol20"] * 100, color="coral")
ax2.set_title("20-Day Annualized Volatility")
ax2.set_ylabel("Volatility (%)")
ax2.tick_params(axis="x", rotation=60)
ax2.grid(True, axis="y", alpha=0.3)

plt.tight_layout()
plt.show()

## Step 4 — Exit Logic

We sell a position if **either** of these is true:
- Its **rank drops below 30** (it's no longer a strong momentum name)
- Its **price drops below the 100-day SMA** (trend is breaking down)

Here we simulate holding the top 40 stocks and check which ones would trigger an exit today.

In [ ]:
# Simulate: pretend we hold the top 40 and see who gets kicked out
sim = scores_df.head(40).copy()
sim["Would_Exit"] = False
sim["Exit_Reason"] = ""

for idx, row in sim.iterrows():
    reasons = []
    if row["Rank"] > ct.EXIT_RANK_CUTOFF:
        reasons.append(f"rank {int(row['Rank'])} > {ct.EXIT_RANK_CUTOFF}")
    if row["Close"] < row["SMA100"]:
        reasons.append("price < 100-SMA")
    if reasons:
        sim.at[idx, "Would_Exit"] = True
        sim.at[idx, "Exit_Reason"] = " & ".join(reasons)

n_exit = sim["Would_Exit"].sum()
print(f"Holding top 40 → {len(sim) - n_exit} HOLD,  {n_exit} EXIT\n")
if n_exit > 0:
    print("Would EXIT:")
    print(sim[sim["Would_Exit"]][["Symbol", "Rank", "Close", "SMA100", "Exit_Reason"]].to_string(index=False))

# Scatter: x = rank, y = price / 100-SMA ratio
# Green dot = hold, Red X = exit
fig, ax = plt.subplots(figsize=(10, 6))
for _, r in sim.iterrows():
    c, m = ("red", "x") if r["Would_Exit"] else ("green", "o")
    ratio = r["Close"] / r["SMA100"]
    ax.scatter(r["Rank"], ratio, color=c, marker=m, s=40)
    if r["Would_Exit"]:
        ax.annotate(r["Symbol"], (r["Rank"], ratio), fontsize=7, alpha=0.7, xytext=(4, 4), textcoords="offset points")

ax.axvline(ct.EXIT_RANK_CUTOFF, color="red", ls="--", alpha=0.5)
ax.axhline(1.0, color="orange", ls="--", alpha=0.5)
ax.set_xlabel("Rank")
ax.set_ylabel("Price / 100-SMA")
ax.set_title("Exit Logic: who stays vs who goes")
ax.legend(handles=[
    Line2D([0], [0], marker="o", color="w", markerfacecolor="green", markersize=8, label="HOLD"),
    Line2D([0], [0], marker="x", color="red", markersize=8, label="EXIT"),
    Line2D([0], [0], ls="--", color="red", alpha=0.5, label=f"Rank cutoff ({ct.EXIT_RANK_CUTOFF})"),
    Line2D([0], [0], ls="--", color="orange", alpha=0.5, label="Price = 100-SMA"),
], loc="upper right")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Step 5 — Deep Dive

A few extra charts to help us understand *why* certain stocks rank high:

- **Log-regression fit** for the top 3 picks — we can visually see how smooth and steep the trend is
- **Score distribution** — how our top picks compare to the rest of the universe
- **Slope vs R²** scatter — are we picking high-slope *and* high-R² names, or just one?
- **Rolling relative strength** vs SPY — are our picks actually outperforming the market?

In [ ]:
# ── Helper: download close prices as a clean 1-D Series ──
def get_close(symbol, period="1y"):
    df = yf.download(symbol, period=period, progress=False)
    close = df["Close"]
    if isinstance(close, pd.DataFrame):  # yfinance sometimes returns multi-index
        close = close.iloc[:, 0]
    return close.dropna()


# ── 5a. Log-regression fit for top 3 picks ──
# We show the top 3 (not just #1) so we can compare how different "strong" trends look
TOP_3 = picks[:3] if len(picks) >= 3 else picks

fig, axes = plt.subplots(1, len(TOP_3), figsize=(5 * len(TOP_3), 4))
if len(TOP_3) == 1:
    axes = [axes]  # make iterable

for ax, sym in zip(axes, TOP_3):
    close = get_close(sym)
    if len(close) < 30:
        ax.set_title(f"{sym}: not enough data")
        continue

    y = np.log(close.values.astype(float))
    x = np.arange(len(y), dtype=float)
    slope, intercept, r_value, _, _ = linregress(x, y)
    r2 = r_value ** 2
    ann_slope = (np.exp(slope) ** 252 - 1) * 100  # annualized %

    ax.plot(close.index, y, label="log(close)")
    ax.plot(close.index, slope * x + intercept, label="fit", ls="--")
    ax.set_title(f"{sym}  (R²={r2:.3f}, slope≈{ann_slope:.0f}%)")
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

fig.suptitle("Log-Price Regression Fit — Top 3 Picks", fontweight="bold")
plt.tight_layout()
plt.show()


# ── 5b. Score distribution: histogram + top-N bar chart ──
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))

ax1.hist(scores_df["Score"].values, bins=50, color="steelblue", edgecolor="white")
ax1.set_title("Score Distribution (full universe)")
ax1.set_xlabel("Clenow Score")
ax1.set_ylabel("Count")
ax1.grid(True, alpha=0.3)

ax2.bar(scores_df.head(TOP_N)["Symbol"], scores_df.head(TOP_N)["Score"], color="steelblue")
ax2.set_title(f"Top {TOP_N} by Score")
ax2.set_ylabel("Score")
ax2.tick_params(axis="x", rotation=60)
ax2.grid(True, axis="y", alpha=0.3)

plt.tight_layout()
plt.show()


# ── 5c. Slope vs R² scatter — are we picking both high-slope AND high-R²? ──
rows = []
for s in scores_df["Symbol"].tolist():
    close = get_close(s)
    close = close.iloc[-90:] if len(close) >= 90 else close
    if len(close) < 30:
        continue
    y = np.log(close.values.astype(float))
    x = np.arange(len(y), dtype=float)
    slope, _, r_val, _, _ = linregress(x, y)
    rows.append({"Symbol": s, "Slope%": (np.exp(slope) ** 252 - 1) * 100, "R²": r_val ** 2})

scatter_df = pd.DataFrame(rows)
if not scatter_df.empty:
    plt.figure(figsize=(7, 5))
    plt.scatter(scatter_df["Slope%"], scatter_df["R²"], s=12, alpha=0.5)
    plt.title("Annualized Slope vs R²")
    plt.xlabel("Annualized Slope (%)")
    plt.ylabel("R²")
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()


# ── 5d. Rolling relative strength vs SPY ──
all_syms = list(dict.fromkeys(picks + ["SPY"]))
rs_data = yf.download(all_syms, period="6mo", progress=False)["Close"].dropna(how="all")

if "SPY" in rs_data.columns:
    bench = rs_data["SPY"]
    plt.figure(figsize=(12, 6))
    for s in picks:
        if s not in rs_data.columns:
            continue
        # rolling 20-day ratio of stock price / SPY price
        ratio = (rs_data[s] / bench).rolling(20).mean()
        plt.plot(ratio.index, ratio.values, label=s)
    plt.title("Rolling 20D Relative Strength vs SPY")
    plt.ylabel("Stock / SPY (rolling mean)")
    plt.legend(ncol=3, fontsize=8)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()